# 02 · Pose inference on REHAB24-6 (SAM 3D Body, MediaPipe)
<a href="https://colab.research.google.com/github/danielamrh/mmc-upper-limb/blob/main/notebooks/02_pose_inference.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab"/></a>

Runs pose models on the Ex1/Ex2 videos (arm abduction, arm VW) of both cameras and caches their raw output
on Drive (`cache/rehab24/<model>/`). Everything is resumable: re-run the notebook after a disconnect and it
continues with the next unfinished 600-frame chunk.

**Runtime: T4 GPU.** MediaPipe runs on the CPU (~7 fps per process); it is faster to run it locally:
`python tools/run_pose.py --root data/rehab24 --cache cache/rehab24 --fetch --model mediapipe-heavy --workers 4`
and copy `cache/rehab24/mediapipe-heavy` to Drive.

**SAM 3D Body needs:** access to the gated checkpoint
[facebook/sam-3d-body-dinov3](https://huggingface.co/facebook/sam-3d-body-dinov3) and a Hugging Face token
stored as Colab secret `HF_TOKEN` (key icon in the left sidebar).

## 0 · Drive + repo + install
Code comes from GitHub (always fresh via `git pull`), data and results live on Google Drive.

In [ ]:
import os, sys
from google.colab import drive
drive.mount("/content/drive")

REPO = "mmc-upper-limb"
REPO_DIR = f"/content/{REPO}"
if not os.path.exists(REPO_DIR):
    !git clone -q https://github.com/danielamrh/{REPO}.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull -q
%cd {REPO_DIR}
COMMIT = !git rev-parse --short HEAD
COMMIT = COMMIT[0]
print("commit:", COMMIT)

!pip install -q -e .
sys.path.insert(0, f"{REPO_DIR}/src")  # editable install is only picked up after a restart
# re-running this cell must pick up freshly pulled code
for name in [m for m in sys.modules if m == "mmcul" or m.startswith("mmcul.")]:
    del sys.modules[name]

DRIVE_ROOT = "/content/drive/MyDrive/mmc_upper_limb"
os.makedirs(DRIVE_ROOT, exist_ok=True)

## 1 · Dependencies
Only the inference dependencies of SAM 3D Body are installed — no detectron2, no renderer: the subject box comes from the dataset and the default field of view is used.

In [ ]:
!pip install -q -e ".[pose]"
!pip install -q pytorch-lightning yacs roma timm einops omegaconf braceexpand huggingface_hub
S3DB = "/content/sam-3d-body"
if not os.path.exists(S3DB):
    !git clone -q https://github.com/facebookresearch/sam-3d-body {S3DB}
sys.path.insert(0, S3DB)
os.environ["PYTHONPATH"] = f"{S3DB}:{REPO_DIR}/src"

from google.colab import userdata
try:
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded")
except (userdata.SecretNotFoundError, userdata.NotebookAccessError) as e:
    raise RuntimeError(
        "Hugging Face token missing: create a 'Read' token at https://huggingface.co/settings/tokens, "
        "add it in Colab's Secrets panel (key icon, left sidebar) under the name HF_TOKEN and switch on "
        "'Notebook access' for this notebook, then re-run this cell.") from e

## 2 · Data
Videos are read from the fast local disk. Only the Ex1/Ex2 members are pulled out of the Zenodo zips (HTTP range requests, ~1.2 GB instead of 2.65 GB).

In [ ]:
DATA = "/content/data/rehab24"
CACHE = f"{DRIVE_ROOT}/cache/rehab24"
!python tools/run_pose.py --root {DATA} --cache {CACHE} --exercises 1 2 --fetch

## 3 · Speed and accuracy test
The same 60-frame clip for every variant: speed, accuracy against mocap (MPJPE after alignment, shoulder
elevation error) and, for fp16, how far it deviates from fp32 with the same backbone. Decides which variant
fits the budget (~74k frames for Ex1, ~165k for Ex1+Ex2, both cameras).

In [ ]:
import time, gc, torch, numpy as np, pandas as pd
from mmcul.datasets import rehab24
from mmcul.pose import make_runner, canonical_pose
from mmcul.align import align_sequence
from mmcul.kinematics import shoulder_elevation
from mmcul.evaluate import ARM_JOINTS
from mmcul.pose.person import bbox_from_2d

A, B = 300, 360
video = rehab24.video_path(DATA, 1, "PM_000", 17)
j2d = np.load(rehab24.joints2d_path(DATA, 1, "PM_000", 17))
boxes =np.stack([bbox_from_2d(j, *rehab24.image_size(17)) for j in j2d[A:B]])
gt = {k: v[A:B] for k, v in rehab24.to_pose(rehab24.load_joints(rehab24.joints_path(DATA, 1, "PM_000"))).items()}

results, poses = [], {}
for name in ["sam3db-vith", "sam3db-vith-fp16", "sam3db-dinov3", "sam3db-dinov3-fp16"]:
    try:
        runner = make_runner(name)
    except Exception as e:  # e.g. access to this checkpoint not granted
        print(name, "failed:", repr(e)[:300]); continue
    runner.run(video, A, A + 2, boxes, 30.0)  # warm-up
    torch.cuda.synchronize(); t0 = time.time()
    seq = runner.run(video, A, B, boxes, 30.0)
    torch.cuda.synchronize(); fps = (B - A) / (time.time() - t0)
    pred = canonical_pose(seq)
    poses[name] = pred
    al = align_sequence(pred, gt, ARM_JOINTS)
    mpjpe = 1000 * np.mean([np.linalg.norm(al[j] - gt[j], axis=1).mean() for j in ARM_JOINTS])
    elev = np.sqrt(np.mean((shoulder_elevation(pred) - shoulder_elevation(gt)) ** 2))
    ref = poses.get(name.removesuffix("-fp16")) if name.endswith("-fp16") else None
    dev = None if ref is None else 1000 * max(np.abs(pred[j] - ref[j]).max() for j in ARM_JOINTS)
    results.append(dict(model=name, fps=fps, h_ex1=73_600 / fps / 3600, h_ex1_ex2=165_000 / fps / 3600,
                        mpjpe_mm=mpjpe, rmse_shoulder_elev=elev, max_dev_vs_fp32_mm=dev))
    del runner; gc.collect(); torch.cuda.empty_cache()
display(pd.DataFrame(results).round(2))

## 4 · Run in the background
The job runs as a separate process (a long cell output crashed the tab in the ACT project); the next cell shows its progress. Choose the model from the speed test.

In [ ]:
MODEL = "sam3db-vith-fp16"  # pick from the speed test
LOG = f"{DRIVE_ROOT}/logs/run_pose_{MODEL}.log"
os.makedirs(os.path.dirname(LOG), exist_ok=True)
!nohup python tools/run_pose.py --root {DATA} --cache {CACHE} --model {MODEL} --exercises 1 2 >> {LOG} 2>&1 &
print("started, log:", LOG)

In [ ]:
!tail -n 5 {LOG}
done = !ls {CACHE}/{MODEL}/Ex*/*.npz 2>/dev/null | wc -l
print(f"{done[0]} of 50 videos finished")